# Train DQN

DQN learns a discrete action-value function from replayed transitions while a target network stabilizes its bootstrap targets. Its one-step target is

$$y_t = r_{t+1} + \gamma(1-d_t) \max_{a'} Q_{\theta^-}(s_{t+1}, a'),$$

where $r_{t+1}$ is the reward, $\gamma$ the discount factor, $d_t$ the terminal indicator, and $Q_{\theta^-}$ the target network evaluated over next actions $a'$. The online network $Q_\theta$ is trained toward $y_t$. This notebook trains DQN on `CartPole-v1`, which has vector observations and two discrete actions, using epsilon-greedy exploration before plotting and evaluating the learned policy.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import DQN, DQNConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "CartPole-v1"

In [ ]:
env = gym.make(ENV_ID)
config = DQNConfig(
    buffer_size=8_000,
    learning_starts=500,
    target_update_interval=250,
    exploration_steps=2_000,
)

agent = DQN(env, config=config, device="cpu")
agent.learn(total_timesteps=10_000)
env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(10, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"DQN training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human")
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")